# 01 — Exploración inicial del dataset

Usa Polars en modo **lazy** para nunca cargar los 9 GB completos en RAM.

In [1]:
import polars as pl
import sys

sys.path.append("../")
from src.abasto.data_loader import scan_ventas, load_catalogo

lf = scan_ventas()
print(lf.collect_schema())

Schema({'tienda_id': Int32, 'articulo_id': Int32, 'fecha': Date, 'unidades_vendidas': Int32})


In [ ]:
# Rango de fechas y cardinalidades (se ejecuta en streaming)
resumen = lf.select([
    pl.col('fecha').min().alias('fecha_min'),
    pl.col('fecha').max().alias('fecha_max'),
    pl.col('tienda_id').n_unique().alias('n_tiendas'),
    pl.col('articulo_id').n_unique().alias('n_articulos'),
    pl.len().alias('n_registros'),
]).collect(engine="streaming")

resumen

In [ ]:
# Muestra las primeras filas
lf.head(10).collect()

tienda_id,articulo_id,fecha,unidades_vendidas
i32,i32,date,i32
1,1,2022-01-03,1
1,10,2022-01-03,1
1,11,2022-01-03,1
1,12,2022-01-03,1
1,15,2022-01-03,1
1,17,2022-01-03,2
1,25,2022-01-03,1
1,27,2022-01-03,1
1,3,2022-01-03,1


In [ ]:
# Nulos por columna
lf.select(pl.all().null_count()).collect(engine="streaming")

tienda_id,articulo_id,fecha,unidades_vendidas
u32,u32,u32,u32
0,0,0,0


In [ ]:
# Estadísticas de unidades vendidas (streaming)
lf.select([
    pl.col('unidades_vendidas').mean().alias('media'),
    pl.col('unidades_vendidas').median().alias('mediana'),
    pl.col('unidades_vendidas').std().alias('std'),
    pl.col('unidades_vendidas').min().alias('min'),
    pl.col('unidades_vendidas').max().alias('max'),
    pl.col('unidades_vendidas').quantile(0.95).alias('p95'),
    pl.col('unidades_vendidas').quantile(0.99).alias('p99'),
]).collect(engine="streaming")

media,mediana,std,min,max,p95,p99
f64,f64,f64,i32,i32,f64,f64
1.825639,1.0,1.476815,1,680,5.0,8.0


In [ ]:
# Catálogo desde Excel
catalogo = load_catalogo()
inv       = catalogo['Inventario'].rename({'Loc': 'tienda_id', 'Sku': 'articulo_id', 'Inventario': 'inventario_inicial'})
cat_sku   = catalogo['CatSku'].rename({'Sku': 'articulo_id', 'Precio': 'precio', 'Costo': 'costo',
                                        'TiempoVida': 'tiempo_vida', 'TamañoSurtido': 'tamano_surtido'})
cat_loc   = catalogo['CatLoc'].rename({'LOC': 'tienda_id', 'REGION': 'region', 'PLAZA': 'plaza'})

print(f"Inventario: {inv.shape}  |  CatSku: {cat_sku.shape}  |  CatLoc: {cat_loc.shape}")
cat_sku

Could not determine dtype for column 2, falling back to string
Could not determine dtype for column 3, falling back to string
Could not determine dtype for column 4, falling back to string
Could not determine dtype for column 5, falling back to string
Could not determine dtype for column 6, falling back to string
Could not determine dtype for column 7, falling back to string
Could not determine dtype for column 8, falling back to string
Could not determine dtype for column 9, falling back to string
Could not determine dtype for column 10, falling back to string
Could not determine dtype for column 11, falling back to string
Could not determine dtype for column 12, falling back to string
Could not determine dtype for column 13, falling back to string


Inventario: (1000000, 3)  |  CatSku: (50, 5)  |  CatLoc: (20000, 3)


articulo_id,precio,costo,tiempo_vida,tamano_surtido
i64,f64,f64,i64,i64
1,41.98,23.7,28,1
2,34.27,22.63,28,1
3,35.43,20.44,21,6
4,34.95,19.91,28,1
5,43.41,27.26,14,1
…,…,…,…,…
46,44.4,28.84,14,4
47,42.47,28.51,28,1
48,23.93,16.01,21,2


## Análisis del catálogo de productos

In [ ]:
# Margen bruto por producto y distribución de TiempoVida / TamañoSurtido
cat_sku_analisis = cat_sku.with_columns([
    ((pl.col('precio') - pl.col('costo')) / pl.col('precio') * 100).alias('margen_pct'),
]).sort('margen_pct', descending=True)

print("Top 5 mayor margen:")
print(cat_sku_analisis.head(5))
print("\nDistribución TiempoVida (días):")
print(cat_sku_analisis['tiempo_vida'].value_counts().sort('tiempo_vida'))
print("\nDistribución TamañoSurtido:")
print(cat_sku_analisis['tamano_surtido'].value_counts().sort('tamano_surtido'))

Top 5 mayor margen:
shape: (5, 6)
┌─────────────┬────────┬───────┬─────────────┬────────────────┬────────────┐
│ articulo_id ┆ precio ┆ costo ┆ tiempo_vida ┆ tamano_surtido ┆ margen_pct │
│ ---         ┆ ---    ┆ ---   ┆ ---         ┆ ---            ┆ ---        │
│ i64         ┆ f64    ┆ f64   ┆ i64         ┆ i64            ┆ f64        │
╞═════════════╪════════╪═══════╪═════════════╪════════════════╪════════════╡
│ 39          ┆ 49.77  ┆ 27.84 ┆ 28          ┆ 2              ┆ 44.062688  │
│ 25          ┆ 30.11  ┆ 16.87 ┆ 21          ┆ 1              ┆ 43.972102  │
│ 34          ┆ 36.37  ┆ 20.47 ┆ 28          ┆ 6              ┆ 43.717349  │
│ 1           ┆ 41.98  ┆ 23.7  ┆ 28          ┆ 1              ┆ 43.544545  │
│ 15          ┆ 34.04  ┆ 19.24 ┆ 10          ┆ 1              ┆ 43.478261  │
└─────────────┴────────┴───────┴─────────────┴────────────────┴────────────┘

Distribución TiempoVida (días):
shape: (4, 2)
┌─────────────┬───────┐
│ tiempo_vida ┆ count │
│ ---         ┆ ---   │


## Integridad: cobertura de pares tienda-artículo

In [ ]:
# Pares únicos tienda-artículo en ventas vs inventario
pares_ventas = (
    lf.select(['tienda_id', 'articulo_id'])
    .unique()
    .collect(engine="streaming")
)
pares_inv = inv.select(['tienda_id', 'articulo_id']).unique()

print(f"Pares en ventas   : {len(pares_ventas):,}")
print(f"Pares en inventario: {len(pares_inv):,}")
print(f"Máximo posible (20k × 50): {20_000 * 50:,}")

# Pares en inventario sin historial de ventas
sin_ventas = pares_inv.join(pares_ventas, on=['tienda_id', 'articulo_id'], how='anti')
print(f"\nPares con inventario pero sin ventas: {len(sin_ventas):,}")

Pares en ventas   : 1,000,000
Pares en inventario: 1,000,000
Máximo posible (20k × 50): 1,000,000

Pares con inventario pero sin ventas: 0


## Semanalización: ventas diarias → semana (lunes)

In [ ]:
# Agrega ventas diarias a semanas ISO (lunes = inicio de semana)
# Se usa streaming para no cargar los 9 GB en RAM
ventas_semana = (
    lf.with_columns(
        # Truncar fecha al lunes de la semana
        pl.col('fecha').dt.truncate('1w').alias('semana')
    )
    .group_by(['tienda_id', 'articulo_id', 'semana'])
    .agg(pl.col('unidades_vendidas').sum().alias('ventas_semana'))
    .sort(['tienda_id', 'articulo_id', 'semana'])
    .collect(engine="streaming")
)

print(f"Filas semanalizadas: {len(ventas_semana):,}")
print(f"Semanas únicas: {ventas_semana['semana'].n_unique()}")
ventas_semana.head(10)

In [ ]:
# Enriquecer con catálogo: precio, costo, tiempo_vida, tamano_surtido
ventas_enriquecida = (
    ventas_semana
    .join(cat_sku.select(['articulo_id', 'precio', 'costo', 'tiempo_vida', 'tamano_surtido']),
          on='articulo_id', how='left')
    .join(inv.select(['tienda_id', 'articulo_id', 'inventario_inicial']),
          on=['tienda_id', 'articulo_id'], how='left')
    .join(cat_loc.select(['tienda_id', 'region', 'plaza']),
          on='tienda_id', how='left')
)

print("Schema enriquecido:")
print(ventas_enriquecida.collect_schema())
ventas_enriquecida.head(5)

In [ ]:
# Guardar el dataset semanal enriquecido como Parquet para los siguientes notebooks
# Parquet es ~10x más compacto que CSV y Polars lo lee mucho más rápido
import os
os.makedirs("../outputs", exist_ok=True)
ventas_enriquecida.write_parquet("../outputs/ventas_semana.parquet", compression="zstd")
print("Guardado en outputs/ventas_semana.parquet")